In [10]:
from pathlib import Path
import pandas as pd

# 1. Xác định thư mục dự án
PROJECT_DIR = next(
    (
        p for p in [Path.cwd(), *Path.cwd().parents]
        if (p / "data/ACME_Insurance").is_dir()
        and (p / "results/llm/reproduction").is_dir()
    ),
    None
)

assert PROJECT_DIR is not None, "Không tìm thấy thư mục dự án"

GEN_DIR = PROJECT_DIR / "results/llm/reproduction"
EVAL_DIR = PROJECT_DIR / "results/evaluation"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

# 2. Kiểm tra đủ 50 file kết quả
run_files = [
    GEN_DIR / f"generation_run_{i:02d}.csv"
    for i in range(1, 51)
]

missing = [f.name for f in run_files if not f.exists()]
assert not missing, f"Thiếu các run: {missing}"

expected_pairs = {
    (qid, method)
    for qid in range(1, 44)
    for method in ("SQL", "SPARQL")
}

required_columns = {
    "ID", "Inquiry URI", "Question",
    "Method", "Generated Query", "Status"
}

# 3. Kiểm tra tính hợp lệ và nhất quán giữa các run
reference_questions = None

for run_id, file in enumerate(run_files, start=1):
    df = pd.read_csv(file, keep_default_na=False)

    assert required_columns.issubset(df.columns), (
        f"{file.name}: thiếu cột bắt buộc"
    )

    # Chuẩn hóa dữ liệu khi kiểm tra, không thay đổi CSV gốc
    df["ID"] = pd.to_numeric(
        df["ID"], errors="raise"
    ).astype(int)

    df["Method"] = df["Method"].astype(str).str.strip().str.upper()
    df["Inquiry URI"] = df["Inquiry URI"].astype(str).str.strip()
    df["Question"] = df["Question"].astype(str).str.strip()
    df["Status"] = df["Status"].astype(str).str.strip()

    pairs = set(zip(df["ID"], df["Method"]))

    assert len(df) == 86, (
        f"{file.name}: số lượng truy vấn không bằng 86"
    )

    assert pairs == expected_pairs, (
        f"{file.name}: sai ID hoặc Method"
    )

    assert not df.duplicated(["ID", "Method"]).any(), (
        f"{file.name}: trùng ID/Method"
    )

    assert df["Status"].eq("GENERATED").all(), (
        f"{file.name}: có truy vấn sinh lỗi"
    )

    assert df["Generated Query"].astype(str).str.strip().ne("").all(), (
        f"{file.name}: có truy vấn rỗng"
    )

    # Kiểm tra cùng 43 câu hỏi và Inquiry URI
    questions = (
        df[["ID", "Method", "Inquiry URI", "Question"]]
        .sort_values(["ID", "Method"])
        .reset_index(drop=True)
    )

    if reference_questions is None:
        reference_questions = questions
    else:
        assert questions.equals(reference_questions), (
            f"{file.name}: câu hỏi hoặc Inquiry URI "
            "không khớp Run 01"
        )

# 4. Tổng hợp
N_RUNS = len(run_files)

assert N_RUNS == 50

print("Project:", PROJECT_DIR)
print("Valid runs:", N_RUNS, "/ 50")
print("SQL queries:", N_RUNS * 43)
print("SPARQL queries:", N_RUNS * 43)
print("Total queries:", N_RUNS * 86)
print("Question consistency: OK")
print("Validation: OK")


Project: c:\Users\Win 11 New\Desktop\thuc-nghiem-co-so-httt
Valid runs: 50 / 50
SQL queries: 2150
SPARQL queries: 2150
Total queries: 4300
Question consistency: OK
Validation: OK


In [11]:
import duckdb
import time
import pandas as pd

# 1. Cấu hình
DB_FILE = PROJECT_DIR / "results/sql/acme.duckdb"
SQL_DIR = EVAL_DIR / "sql_runs"
SQL_DIR.mkdir(parents=True, exist_ok=True)

assert DB_FILE.exists(), f"Không tìm thấy database: {DB_FILE}"

# 2. Đánh giá SQL của 50 run
for file in run_files:
    run_id = file.stem.split("_")[-1]
    run_dir = SQL_DIR / f"run_{run_id}"
    run_dir.mkdir(parents=True, exist_ok=True)

    df = pd.read_csv(file, keep_default_na=False)

    sql_df = df[
        df["Method"].astype(str).str.strip().str.upper() == "SQL"
    ].sort_values("ID")

    assert len(sql_df) == 43, f"Run {run_id}: thiếu SQL"

    log_file = run_dir / "execution_log.csv"
    records = {}

    # Đọc checkpoint đã có
    if log_file.exists():
        old = pd.read_csv(log_file, keep_default_na=False)

        assert {"ID", "Status", "Error", "Time"}.issubset(old.columns)
        assert not old["ID"].duplicated().any()

        records = {
            int(row["ID"]): row.to_dict()
            for _, row in old.iterrows()
        }

    reused = 0
    con = None

    try:
        for _, row in sql_df.iterrows():
            qid = int(row["ID"])
            query = str(row["Generated Query"]).strip()
            output = run_dir / f"{qid:02d}.csv"

            # Giữ kết quả đã hoàn thành
            if qid in records:
                old_status = records[qid]["Status"]

                if old_status == "OK" and output.exists():
                    reused += 1
                    continue

                if old_status in ("ERROR", "TIMEOUT"):
                    reused += 1
                    continue

            # Chỉ mở kết nối nếu cần thực thi
            if con is None:
                con = duckdb.connect(str(DB_FILE), read_only=True)

            start = time.perf_counter()
            status = "OK"
            error = ""

            try:
                # Chỉ chấp nhận truy vấn SELECT/WITH
                if not query.upper().startswith(("SELECT", "WITH")):
                    raise ValueError("Not a SELECT/WITH query")

                result = con.execute(query).fetchdf()
                result.to_csv(output, index=False)

            except Exception as e:
                status = "ERROR"
                error = str(e)
                output.unlink(missing_ok=True)

            records[qid] = {
                "ID": qid,
                "Status": status,
                "Error": error,
                "Time": round(time.perf_counter() - start, 3)
            }

            # Lưu checkpoint sau mỗi truy vấn
            pd.DataFrame(
                sorted(records.values(), key=lambda r: int(r["ID"]))
            ).to_csv(
                log_file,
                index=False,
                encoding="utf-8-sig"
            )

    finally:
        if con is not None:
            con.close()

    # 3. Kiểm tra kết quả từng run
    log_df = pd.DataFrame(records.values())

    assert len(log_df) == 43, f"Run {run_id}: thiếu kết quả"
    assert set(log_df["ID"].astype(int)) == set(range(1, 44))

    print(
        f"Run {run_id}:",
        log_df["Status"].value_counts().to_dict(),
        f"| Reused: {reused}/43"
    )

print("\nSQL execution completed.")
print("Total runs:", len(run_files))
print("Total SQL queries:", len(run_files) * 43)


Run 01: {'OK': 42, 'ERROR': 1} | Reused: 43/43
Run 02: {'OK': 42, 'ERROR': 1} | Reused: 43/43
Run 03: {'OK': 40, 'ERROR': 3} | Reused: 43/43
Run 04: {'OK': 42, 'ERROR': 1} | Reused: 43/43
Run 05: {'OK': 40, 'ERROR': 3} | Reused: 43/43
Run 06: {'OK': 41, 'ERROR': 2} | Reused: 43/43
Run 07: {'OK': 42, 'ERROR': 1} | Reused: 43/43
Run 08: {'OK': 41, 'ERROR': 2} | Reused: 43/43
Run 09: {'OK': 42, 'ERROR': 1} | Reused: 43/43
Run 10: {'OK': 42, 'ERROR': 1} | Reused: 43/43
Run 11: {'OK': 41, 'ERROR': 2} | Reused: 43/43
Run 12: {'OK': 42, 'ERROR': 1} | Reused: 43/43
Run 13: {'OK': 37, 'ERROR': 6} | Reused: 43/43
Run 14: {'OK': 42, 'ERROR': 1} | Reused: 43/43
Run 15: {'OK': 42, 'ERROR': 1} | Reused: 43/43
Run 16: {'OK': 42, 'ERROR': 1} | Reused: 43/43
Run 17: {'OK': 42, 'ERROR': 1} | Reused: 43/43
Run 18: {'OK': 42, 'ERROR': 1} | Reused: 43/43
Run 19: {'OK': 42, 'ERROR': 1} | Reused: 43/43
Run 20: {'OK': 41, 'ERROR': 2} | Reused: 43/43
Run 21: {'OK': 42, 'ERROR': 1} | Reused: 43/43
Run 22: {'OK'

In [12]:
import subprocess
import re
import time
import hashlib
import pandas as pd
from pathlib import Path

# 1. Cấu hình Ontop
ONTOP = Path.home() / "tools/ontop/ontop.bat"
MAPPING = PROJECT_DIR / "results/ontop/acme_mapping.ttl"
ONTOLOGY = PROJECT_DIR / "data/ACME_Insurance/ontology/insurance.ttl"
PROPERTIES = PROJECT_DIR / "results/ontop/duckdb.properties"

SPARQL_DIR = EVAL_DIR / "sparql_runs"
SPARQL_DIR.mkdir(parents=True, exist_ok=True)

for path in [ONTOP, MAPPING, ONTOLOGY, PROPERTIES]:
    assert path.exists(), f"Không tìm thấy: {path}"

def sha256_file(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

mapping_hash = sha256_file(MAPPING)
ontology_hash = sha256_file(ONTOLOGY)
properties_hash = sha256_file(PROPERTIES)

# 2. Chuyển SERVICE :mapped sang truy vấn Ontop local
service_pattern = re.compile(
    r"\bSERVICE\s+(?:SILENT\s+)?"
    r"(?P<target><[^>]+>|(?:[A-Za-z_][\w.-]*)?:[\w.-]+)"
    r"\s*(?=\{)",
    flags=re.IGNORECASE
)

def prepare_local_query(raw_query):
    def remove_service(match):
        target = match.group("target")

        if "mapped" not in target.lower():
            raise ValueError(
                f"SERVICE ngoài phạm vi KG local: {target}"
            )

        return ""

    query = service_pattern.sub(remove_service, raw_query)

    # Không tự ý loại bỏ SERVICE khác
    if re.search(r"\bSERVICE\s+", query, re.IGNORECASE):
        raise ValueError("Truy vấn còn SERVICE chưa hỗ trợ")

    return query.strip()


# 3. Thực thi 50 run
for file in run_files:
    run_id = file.stem.split("_")[-1]
    run_dir = SPARQL_DIR / f"run_{run_id}"
    run_dir.mkdir(parents=True, exist_ok=True)

    queries_dir = run_dir / "queries"
    queries_dir.mkdir(exist_ok=True)

    df = pd.read_csv(file, keep_default_na=False)

    queries = df[
        df["Method"].astype(str).str.strip().str.upper() == "SPARQL"
    ].sort_values("ID")

    assert len(queries) == 43, f"Run {run_id}: thiếu SPARQL"

    
    log_file = run_dir / "execution_log.csv"
    records = {}

    # Chỉ đọc checkpoint nếu file không rỗng
    if log_file.exists() and log_file.stat().st_size > 0:
        try:
            old = pd.read_csv(log_file, keep_default_na=False)

            assert {"ID", "Status"}.issubset(old.columns)
            assert not old["ID"].duplicated().any()

            records = {
                int(r["ID"]): r.to_dict()
                for _, r in old.iterrows()
            }

        except pd.errors.EmptyDataError:
            print(f"Run {run_id}: checkpoint rỗng")
    else:
        print(f"Run {run_id}: chưa có checkpoint hợp lệ")

    # Khôi phục kết quả đã thực thi thành công nhưng mất checkpoint
    recovered = 0

    for _, row in queries.iterrows():
        qid = int(row["ID"])

        if qid in records:
            continue

        raw_query = str(row["Generated Query"]).strip()

        original_file = queries_dir / f"{qid:02d}_original.rq"
        query_file = queries_dir / f"{qid:02d}_ontop.rq"
        output_file = run_dir / f"{qid:02d}.csv"

        if not all(p.exists() for p in [
            original_file, query_file, output_file
        ]):
            continue

        try:
            local_query = prepare_local_query(raw_query)

            if original_file.read_text(encoding="utf-8") != raw_query:
                continue

            if query_file.read_text(encoding="utf-8") != local_query:
                continue

            pd.read_csv(output_file)

            records[qid] = {
                "ID": qid,
                "Status": "OK",
                "Error": "",
                "Time (s)": "",
                "Query SHA256": hashlib.sha256(
                    local_query.encode("utf-8")
                ).hexdigest(),
                "Mapping SHA256": mapping_hash,
                "Ontology SHA256": ontology_hash,
                "Properties SHA256": properties_hash
            }

            recovered += 1

        except (ValueError, OSError, pd.errors.ParserError,
                pd.errors.EmptyDataError):
            continue

    if recovered:
        print(f"Run {run_id}: recovered {recovered} OK results")

    reused = 0

    for _, row in queries.iterrows():
        qid = int(row["ID"])
        raw_query = str(row["Generated Query"]).strip()

        original_file = queries_dir / f"{qid:02d}_original.rq"
        query_file = queries_dir / f"{qid:02d}_ontop.rq"
        output_file = run_dir / f"{qid:02d}.csv"
        error_log = run_dir / f"{qid:02d}.log"

        start = time.perf_counter()
        status = "ERROR"
        error = ""
        query_hash = ""

        try:
            local_query = prepare_local_query(raw_query)

            query_hash = hashlib.sha256(
                local_query.encode("utf-8")
            ).hexdigest()

            # 4. Kiểm tra checkpoint
            previous = records.get(qid)

            if previous and previous.get("Status") == "OK":
                hashes_match = (
                    previous.get("Query SHA256") == query_hash
                    and previous.get("Mapping SHA256") == mapping_hash
                    and previous.get("Ontology SHA256") == ontology_hash
                    and previous.get("Properties SHA256") == properties_hash
                )

                # Tương thích log cũ chưa có hash.
                # Chỉ tái sử dụng khi file truy vấn gốc và
                # truy vấn local còn giống hệt nhau.
                legacy_match = (
                    not previous.get("Query SHA256")
                    and original_file.exists()
                    and query_file.exists()
                    and original_file.read_text(
                        encoding="utf-8"
                    ) == raw_query
                    and query_file.read_text(
                        encoding="utf-8"
                    ) == local_query
                )

                if (
                    output_file.exists()
                    and (hashes_match or legacy_match)
                ):
                    try:
                        pd.read_csv(output_file)
                        reused += 1
                        continue
                    except Exception:
                        pass

            # 5. Lưu truy vấn
            original_file.write_text(
                raw_query, encoding="utf-8"
            )

            query_file.write_text(
                local_query, encoding="utf-8"
            )

            output_file.unlink(missing_ok=True)

            # 6. Thực thi Ontop
            cmd = [
                str(ONTOP), "query",
                "-m", MAPPING.resolve().as_uri(),
                "-t", str(ONTOLOGY.resolve()),
                "-p", PROPERTIES.resolve().as_uri(),
                "-q", str(query_file.resolve()),
                "-o", str(output_file.resolve())
            ]

            process = subprocess.run(
                cmd,
                cwd=ONTOP.parent,
                capture_output=True,
                text=True,
                encoding="utf-8",
                errors="replace",
                timeout=60
            )

            log = process.stderr + "\n" + process.stdout
            error_log.write_text(log, encoding="utf-8")

            if process.returncode != 0 or not output_file.exists():
                raise RuntimeError(
                    log[-1500:] or "Ontop không tạo CSV"
                )

            if any(x in log for x in [
                "QueryEvaluationException",
                "RepositoryException",
                "Exception in thread"
            ]):
                raise RuntimeError(log[-1500:])

            # Kiểm tra CSV thực sự đọc được
            pd.read_csv(output_file)

            status = "OK"

        except subprocess.TimeoutExpired:
            status = "TIMEOUT"
            error = "Exceeded 60 seconds"

        except Exception as e:
            status = "ERROR"
            error = f"{type(e).__name__}: {e}"[-1500:]

        if status != "OK":
            output_file.unlink(missing_ok=True)

        # 7. Lưu checkpoint sau mỗi truy vấn
        records[qid] = {
            "ID": qid,
            "Status": status,
            "Error": error,
            "Time (s)": round(
                time.perf_counter() - start, 3
            ),
            "Query SHA256": query_hash,
            "Mapping SHA256": mapping_hash,
            "Ontology SHA256": ontology_hash,
            "Properties SHA256": properties_hash
        }


        # Ghi checkpoint an toàn để tránh file 0 byte khi crash
        tmp_file = log_file.with_suffix(".tmp")

        pd.DataFrame(
            sorted(records.values(), key=lambda x: int(x["ID"]))
        ).to_csv(
            tmp_file,
            index=False,
            encoding="utf-8-sig"
        )

        tmp_file.replace(log_file)


    # 8. Tổng kết mỗi run
    log_df = pd.read_csv(log_file, keep_default_na=False)

    assert len(log_df) == 43
    assert log_df["ID"].nunique() == 43

    print(
        f"Run {run_id}:",
        log_df["Status"].value_counts().to_dict(),
        f"| Reused: {reused}"
    )

print("SPARQL execution completed.")


Run 01: {'OK': 36, 'ERROR': 7} | Reused: 36
Run 02: {'OK': 37, 'ERROR': 6} | Reused: 37
Run 03: {'OK': 35, 'ERROR': 8} | Reused: 35
Run 04: {'OK': 39, 'ERROR': 4} | Reused: 39
Run 05: {'OK': 37, 'ERROR': 6} | Reused: 37
Run 06: {'OK': 37, 'ERROR': 6} | Reused: 37
Run 07: {'OK': 35, 'ERROR': 8} | Reused: 35
Run 08: {'OK': 39, 'ERROR': 4} | Reused: 39
Run 09: {'OK': 38, 'ERROR': 5} | Reused: 38
Run 10: {'OK': 37, 'ERROR': 6} | Reused: 37
Run 11: {'OK': 36, 'ERROR': 7} | Reused: 36
Run 12: {'OK': 38, 'ERROR': 5} | Reused: 38
Run 13: {'OK': 38, 'ERROR': 5} | Reused: 38
Run 14: {'OK': 36, 'ERROR': 7} | Reused: 36
Run 15: {'OK': 38, 'ERROR': 5} | Reused: 38
Run 16: {'OK': 36, 'ERROR': 7} | Reused: 36
Run 17: {'OK': 37, 'ERROR': 6} | Reused: 37
Run 18: {'OK': 39, 'ERROR': 4} | Reused: 39
Run 19: {'OK': 37, 'ERROR': 6} | Reused: 37
Run 20: {'OK': 39, 'ERROR': 4} | Reused: 39
Run 21: {'OK': 36, 'ERROR': 7} | Reused: 36
Run 22: {'OK': 35, 'ERROR': 8} | Reused: 35
Run 23: {'OK': 36, 'ERROR': 7} |

In [14]:
import pandas as pd

# 1. Xác định thư mục Reference
SQL_REF_DIR = PROJECT_DIR / "results/sql/reference_results"
ONTOP_DIR = PROJECT_DIR / "results/ontop"
SPARQL_REF_DIR = ONTOP_DIR / "reference_results"

for name, folder in [
    ("SQL", SQL_REF_DIR),
    ("SPARQL", SPARQL_REF_DIR)
]:
    assert folder.is_dir(), f"Không tìm thấy: {folder}"

    files = sorted(folder.glob("*.csv"))
    print(f"\n{name} Reference: {len(files)} files")
    print("Examples:", [f.name for f in files[:10]])

# 2. Đọc bảng ánh xạ Reference
comparison_file = ONTOP_DIR / "reference_comparison.csv"
assert comparison_file.exists(), f"Không tìm thấy: {comparison_file}"

reference = pd.read_csv(
    comparison_file,
    keep_default_na=False
)

required_columns = {"ID", "SQL Reference"}
assert required_columns.issubset(reference.columns)

reference["ID"] = reference["ID"].astype(int)

assert len(reference) == 43
assert reference["ID"].is_unique
assert set(reference["ID"]) == set(range(1, 44))

print("\n=== REFERENCE MAPPING ===")
print("Columns:", reference.columns.tolist())
print("Rows:", len(reference))

display(reference.head(5))

# 3. Kiểm tra đủ Reference cho 43 câu hỏi
missing_sql = []
missing_sparql = []

for _, row in reference.iterrows():
    qid = int(row["ID"])

    sql_name = (
        str(row["SQL Reference"])
        .rstrip("/")
        .split("/")[-1] + ".csv"
    )
    sql_file = SQL_REF_DIR / sql_name

    sparql_name = (
        "18_ontop.csv" if qid == 18
        else f"{qid:02d}.csv"
    )
    sparql_file = SPARQL_REF_DIR / sparql_name

    if not sql_file.exists():
        missing_sql.append(qid)

    if not sparql_file.exists():
        missing_sparql.append(qid)

assert not missing_sql, f"Thiếu SQL Reference: {missing_sql}"
assert not missing_sparql, f"Thiếu SPARQL Reference: {missing_sparql}"

# 4. Kiểm tra ngoại lệ Question 18
print("\nQuestion 18 Reference:")
print(
    "18_ontop.csv exists:",
    (SPARQL_REF_DIR / "18_ontop.csv").exists()
)

# 5. Tổng kết
print("\n=== REFERENCE VALIDATION ===")
print("Questions:", len(reference))
print("SQL References: 43/43 OK")
print("SPARQL References: 43/43 OK")
print("Validation: OK")



SQL Reference: 45 files
Examples: ['query-08e96beb-2717-4045-a830-43749d3cd18c.csv', 'query-09478b45-7383-4608-b696-26f2f6bbe383.csv', 'query-0a848d91-fc89-4774-ae38-3b1015a1dc42.csv', 'query-244778fe-5504-4639-b174-37f013c52700.csv', 'query-366dfab8-1aea-4d48-ac19-84b98432b89d.csv', 'query-37e3d4bb-7132-47a6-a008-47a37cdb9d2c.csv', 'query-4080805b-3146-45f9-8325-51e2d3c00e8d.csv', 'query-40c8a2e3-8530-4b48-8b03-6364070baa32.csv', 'query-425d1228-4470-4470-a85e-26d5003a1990.csv', 'query-453d5cbf-8fa3-46b2-83d5-820fac1a583c.csv']

SPARQL Reference: 43 files
Examples: ['01.csv', '02.csv', '03.csv', '04.csv', '05.csv', '06.csv', '07.csv', '08.csv', '09.csv', '10.csv']

=== REFERENCE MAPPING ===
Columns: ['ID', 'Status', 'SQL Reference', 'SPARQL Rows']
Rows: 43


,ID,Status,SQL Reference,SPARQL Rows
0,1,MATCH,https://templates.data.world/query-63dd7451-59...,1
1,2,MATCH,https://templates.data.world/query-5e26c413-d1...,2
2,3,MATCH,https://templates.data.world/query-47603eb6-8f...,2
3,4,MATCH,https://templates.data.world/query-37e3d4bb-71...,1
4,5,MATCH,https://templates.data.world/query-425d1228-44...,2



Question 18 Reference:
18_ontop.csv exists: True

=== REFERENCE VALIDATION ===
Questions: 43
SQL References: 43/43 OK
SPARQL References: 43/43 OK
Validation: OK


In [15]:
from collections import Counter
from decimal import Decimal
import re
import pandas as pd

# 1. Chuẩn hóa giá trị trước khi so sánh
def normalize(value):
    value = str(value).strip()

    if value == "":
        return ""

    # 20.50 và 20.5 được coi là bằng nhau
    try:
        if re.fullmatch(r"[+-]?\d+(\.\d+)?([eE][+-]?\d+)?", value):
            return str(Decimal(value).normalize())
    except Exception:
        pass

    return value


# 2. So sánh hai bảng kết quả
def same_result(expected_file, actual_file):
    a = pd.read_csv(
        expected_file,
        dtype=str,
        keep_default_na=False
    )

    b = pd.read_csv(
        actual_file,
        dtype=str,
        keep_default_na=False
    )

    if a.shape != b.shape:
        return False

    a_rows = [
        tuple(map(normalize, row))
        for row in a.itertuples(index=False, name=None)
    ]

    b_rows = [
        tuple(map(normalize, row))
        for row in b.itertuples(index=False, name=None)
    ]

    target = Counter(a_rows)
    n_cols = a.shape[1]

    # Chỉ thử các cách đổi cột có cùng tập giá trị
    candidates = []

    for i in range(n_cols):
        values_a = Counter(row[i] for row in a_rows)

        candidates.append([
            j for j in range(n_cols)
            if values_a == Counter(row[j] for row in b_rows)
        ])

    if any(not options for options in candidates):
        return False

    order = sorted(
        range(n_cols),
        key=lambda i: len(candidates[i])
    )

    mapping = [None] * n_cols
    used = set()

    def check(k):
        if k == n_cols:
            rows = Counter(
                tuple(row[mapping[i]] for i in range(n_cols))
                for row in b_rows
            )
            return rows == target

        i = order[k]

        for j in candidates[i]:
            if j not in used:
                mapping[i] = j
                used.add(j)

                if check(k + 1):
                    return True

                used.remove(j)

        return False

    return check(0)


# 3. Ánh xạ SQL Reference theo ID câu hỏi
sql_refs = {}

for _, row in reference.iterrows():
    qid = int(row["ID"])

    filename = (
        str(row["SQL Reference"])
        .rstrip("/")
        .split("/")[-1] + ".csv"
    )

    sql_refs[qid] = SQL_REF_DIR / filename

assert len(sql_refs) == 43
assert set(sql_refs) == set(range(1, 44))


# 4. Chấm toàn bộ 50 run
records = []

for file in run_files:
    run_id = int(file.stem.split("_")[-1])

    print(f"Evaluating Run {run_id:02d}...", flush=True)

    for method in ["SQL", "SPARQL"]:
        folder = (
            EVAL_DIR
            / f"{method.lower()}_runs"
            / f"run_{run_id:02d}"
        )

        log_file = folder / "execution_log.csv"

        # Không chấm khi checkpoint chưa hoàn chỉnh
        assert log_file.exists(), (
            f"Run {run_id:02d} {method}: thiếu execution_log.csv"
        )

        assert log_file.stat().st_size > 0, (
            f"Run {run_id:02d} {method}: checkpoint rỗng"
        )

        log = pd.read_csv(
            log_file,
            keep_default_na=False
        )

        assert {"ID", "Status"}.issubset(log.columns), (
            f"Run {run_id:02d} {method}: log thiếu cột"
        )

        log["ID"] = log["ID"].astype(int)

        assert len(log) == 43, (
            f"Run {run_id:02d} {method}: "
            f"chỉ có {len(log)}/43 kết quả"
        )

        assert not log["ID"].duplicated().any(), (
            f"Run {run_id:02d} {method}: trùng ID"
        )

        assert set(log["ID"]) == set(range(1, 44)), (
            f"Run {run_id:02d} {method}: sai danh sách ID"
        )

        log = log.set_index("ID")

        for qid in range(1, 44):
            # Xác định Reference
            if method == "SQL":
                gold = sql_refs[qid]
            else:
                name = (
                    "18_ontop.csv"
                    if qid == 18
                    else f"{qid:02d}.csv"
                )

                gold = SPARQL_REF_DIR / name

            actual = folder / f"{qid:02d}.csv"
            status = str(log.loc[qid, "Status"]).strip()

            if not gold.exists():
                raise FileNotFoundError(gold)

            # ERROR/TIMEOUT được tính là không chính xác
            correct = (
                status == "OK"
                and actual.exists()
                and same_result(gold, actual)
            )

            records.append({
                "Run": run_id,
                "ID": qid,
                "Method": method,
                "Status": status,
                "Correct": int(correct)
            })


# 5. Kiểm tra kết quả đánh giá
ea_df = pd.DataFrame(records)

assert len(ea_df) == N_RUNS * 86, (
    f"Thiếu kết quả: {len(ea_df)}/{N_RUNS * 86}"
)

assert not ea_df.duplicated(
    ["Run", "ID", "Method"]
).any(), "Kết quả đánh giá bị trùng"

counts = ea_df.groupby(["Run", "Method"]).size()

assert len(counts) == N_RUNS * 2
assert counts.eq(43).all(), (
    "Có run không đủ 43 truy vấn SQL/SPARQL"
)


# 6. Lưu Execution Accuracy
output_file = EVAL_DIR / "execution_accuracy_all_runs.csv"

ea_df.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

summary = ea_df.groupby(
    ["Run", "Method"]
)["Correct"].agg(
    Correct="sum",
    EA="mean"
)

summary["EA"] = (
    summary["EA"] * 100
).round(2)


# 7. Hiển thị kết quả
print("\n=== EXECUTION ACCURACY (%) ===")
display(summary)

print("\nTotal runs:", N_RUNS)
print("Total evaluated:", len(ea_df))
print("SQL evaluated:", (ea_df["Method"] == "SQL").sum())
print("SPARQL evaluated:", (ea_df["Method"] == "SPARQL").sum())

print("\nSaved:", output_file)


Evaluating Run 01...
Evaluating Run 02...
Evaluating Run 03...
Evaluating Run 04...
Evaluating Run 05...
Evaluating Run 06...
Evaluating Run 07...
Evaluating Run 08...
Evaluating Run 09...
Evaluating Run 10...
Evaluating Run 11...
Evaluating Run 12...
Evaluating Run 13...
Evaluating Run 14...
Evaluating Run 15...
Evaluating Run 16...
Evaluating Run 17...
Evaluating Run 18...
Evaluating Run 19...
Evaluating Run 20...
Evaluating Run 21...
Evaluating Run 22...
Evaluating Run 23...
Evaluating Run 24...
Evaluating Run 25...
Evaluating Run 26...
Evaluating Run 27...
Evaluating Run 28...
Evaluating Run 29...
Evaluating Run 30...
Evaluating Run 31...
Evaluating Run 32...
Evaluating Run 33...
Evaluating Run 34...
Evaluating Run 35...
Evaluating Run 36...
Evaluating Run 37...
Evaluating Run 38...
Evaluating Run 39...
Evaluating Run 40...
Evaluating Run 41...
Evaluating Run 42...
Evaluating Run 43...
Evaluating Run 44...
Evaluating Run 45...
Evaluating Run 46...
Evaluating Run 47...
Evaluating Ru

Correct     EA
Run Method                
1   SPARQL       26  60.47
    SQL          18  41.86
2   SPARQL       26  60.47
    SQL          18  41.86
3   SPARQL       25  58.14
...             ...    ...
48  SQL          20  46.51
49  SPARQL       30  69.77
    SQL          20  46.51
50  SPARQL       30  69.77
    SQL          19  44.19

[100 rows x 2 columns]


Total runs: 50
Total evaluated: 4300
SQL evaluated: 2150
SPARQL evaluated: 2150

Saved: c:\Users\Win 11 New\Desktop\thuc-nghiem-co-so-httt\results\evaluation\execution_accuracy_all_runs.csv


In [16]:
import pandas as pd

# 1. Đọc kết quả Execution Accuracy
ea_file = EVAL_DIR / "execution_accuracy_all_runs.csv"
assert ea_file.exists(), f"Không tìm thấy: {ea_file}"

ea_df = pd.read_csv(ea_file)

ea_df["Run"] = ea_df["Run"].astype(int)
ea_df["ID"] = ea_df["ID"].astype(int)
ea_df["Correct"] = ea_df["Correct"].astype(int)

# Kiểm tra đủ 50 run, mỗi run gồm 43 SQL và 43 SPARQL
assert len(ea_df) == 4300
assert set(ea_df["Run"]) == set(range(1, 51))
assert set(ea_df["ID"]) == set(range(1, 44))
assert set(ea_df["Method"]) == {"SQL", "SPARQL"}
assert ea_df["Correct"].isin([0, 1]).all()
assert not ea_df.duplicated(["Run", "ID", "Method"]).any()

counts = ea_df.groupby(["ID", "Method"]).size()
assert len(counts) == 86
assert counts.eq(50).all()

print("Validated:", len(ea_df), "evaluation records")

# 2. OEA: tỷ lệ trả lời đúng của từng câu hỏi qua 50 run
oea_df = (
    ea_df.groupby(["ID", "Method"], as_index=False)["Correct"]
    .mean()
    .rename(columns={"Correct": "OEA"})
)

# 3. AOEA: trung bình OEA trên 43 câu hỏi
aoea_df = (
    oea_df.groupby("Method", as_index=False)["OEA"]
    .mean()
    .rename(columns={"OEA": "AOEA"})
)

# 4. Lưu kết quả với giá trị gốc từ 0 đến 1
oea_df.to_csv(
    EVAL_DIR / "oea_per_question.csv",
    index=False,
    encoding="utf-8-sig"
)

aoea_df.to_csv(
    EVAL_DIR / "aoea_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

# 5. Hiển thị AOEA dưới dạng phần trăm
print("\n=== AOEA (%) ===")
display(
    aoea_df.assign(
        AOEA=lambda x: (x["AOEA"] * 100).round(2)
    )
)

# 6. Hiển thị OEA theo từng câu hỏi
print("\n=== OEA PER QUESTION (%) ===")
display(
    oea_df.assign(
        OEA=lambda x: (x["OEA"] * 100).round(2)
    )
)

print("\nQuestions:", oea_df["ID"].nunique())
print("Runs per question:", 50)
print("Saved: oea_per_question.csv, aoea_summary.csv")


Validated: 4300 evaluation records

=== AOEA (%) ===


,Method,AOEA
0,SPARQL,65.53
1,SQL,45.72



=== OEA PER QUESTION (%) ===


,ID,Method,OEA
0,1,SPARQL,0.0
1,1,SQL,0.0
2,2,SPARQL,100.0
3,2,SQL,100.0
4,3,SPARQL,20.0
...,...,...,...
81,41,SQL,94.0
82,42,SPARQL,100.0
83,42,SQL,100.0
84,43,SPARQL,100.0



Questions: 43
Runs per question: 50
Saved: oea_per_question.csv, aoea_summary.csv


In [17]:
import re
import pandas as pd
from rdflib import Graph, Namespace, RDF
from rdflib.namespace import DCTERMS

# 1. Đọc OEA đã tính từ 50 run
oea_file = EVAL_DIR / "oea_per_question.csv"
assert oea_file.exists(), f"Không tìm thấy: {oea_file}"

oea_df = pd.read_csv(oea_file)
oea_df["ID"] = oea_df["ID"].astype(int)

assert len(oea_df) == 86
assert not oea_df.duplicated(["ID", "Method"]).any()

# 2. Đọc benchmark gốc
benchmark = Graph()
benchmark.parse(
    PROJECT_DIR / "data/ACME_Insurance/investigation/acme-benchmark.ttl",
    format="turtle"
)

QA = Namespace("http://models.data.world/benchmarks/QandA#")

# 3. Lấy Quadrant theo đúng thứ tự ID câu hỏi
rows = []

extra_question = (
    "Return all the claims by claim number "
    "that were closed in 2019"
)

for inquiry in sorted(
    benchmark.subjects(RDF.type, QA.Inquiry),
    key=str
):
    question = str(benchmark.value(inquiry, QA.prompt))

    if question == extra_question:
        continue

    labels = set()

    for ref in benchmark.objects(inquiry, QA.expects):
        title = str(
            benchmark.value(ref, DCTERMS.title) or ""
        )

        match = re.match(
            r"(LQLS|HQLS|LQHS|HQHS)\b",
            title.strip(),
            re.IGNORECASE
        )

        if match:
            labels.add(match.group(1).upper())

    assert len(labels) == 1, (
        f"Quadrant không xác định: {question}"
    )

    rows.append({
        "ID": len(rows) + 1,
        "Quadrant": labels.pop()
    })

quadrants = pd.DataFrame(rows)

assert len(quadrants) == 43
assert quadrants["ID"].is_unique
assert set(quadrants["ID"]) == set(range(1, 44))

print("=== QUESTION DISTRIBUTION ===")
print(quadrants["Quadrant"].value_counts().sort_index())

# 4. Ghép Quadrant với OEA
oea_by_quadrant = oea_df.merge(
    quadrants,
    on="ID",
    how="inner",
    validate="many_to_one"
)

assert len(oea_by_quadrant) == 86

# 5. Tính AOEA theo Quadrant và phương pháp
quadrant_aoea = (
    oea_by_quadrant
    .groupby(["Quadrant", "Method"])["OEA"]
    .mean()
    .reset_index(name="AOEA")
)

# 6. Lưu kết quả
oea_by_quadrant.to_csv(
    EVAL_DIR / "oea_by_quadrant.csv",
    index=False,
    encoding="utf-8-sig"
)

quadrant_aoea.to_csv(
    EVAL_DIR / "aoea_by_quadrant.csv",
    index=False,
    encoding="utf-8-sig"
)

# 7. Hiển thị kết quả
print("\n=== AOEA BY QUADRANT (%) ===")

display(
    quadrant_aoea.assign(
        AOEA=lambda x: (x["AOEA"] * 100).round(2)
    )
)

print("\nQuestions:", len(quadrants))
print("OEA records:", len(oea_by_quadrant))
print("Quadrants:", quadrants["Quadrant"].nunique())

print("\nSaved:")
print("oea_by_quadrant.csv")
print("aoea_by_quadrant.csv")


=== QUESTION DISTRIBUTION ===
Quadrant
HQHS    10
HQLS    11
LQHS    10
LQLS    12
Name: count, dtype: int64

=== AOEA BY QUADRANT (%) ===


,Quadrant,Method,AOEA
0,HQHS,SPARQL,44.20
1,HQHS,SQL,49.60
2,HQLS,SPARQL,66.91
3,HQLS,SQL,52.73
4,LQHS,SPARQL,67.20
5,LQHS,SQL,9.80
6,LQLS,SPARQL,80.67
7,LQLS,SQL,66.00



Questions: 43
OEA records: 86
Quadrants: 4

Saved:
oea_by_quadrant.csv
aoea_by_quadrant.csv


In [18]:
import pandas as pd
from rdflib import Graph, Namespace, RDF

# 1. Đọc 43 câu hỏi từ benchmark gốc
benchmark = Graph()
benchmark.parse(
    PROJECT_DIR / "data/ACME_Insurance/investigation/acme-benchmark.ttl",
    format="turtle"
)

QA = Namespace("http://models.data.world/benchmarks/QandA#")

extra_question = (
    "Return all the claims by claim number "
    "that were closed in 2019"
)

benchmark_questions = []

for inquiry in sorted(
    benchmark.subjects(RDF.type, QA.Inquiry),
    key=str
):
    question = str(benchmark.value(inquiry, QA.prompt))

    if question == extra_question:
        continue

    benchmark_questions.append({
        "ID": len(benchmark_questions) + 1,
        "Benchmark URI": str(inquiry),
        "Benchmark Question": question
    })

expected = pd.DataFrame(benchmark_questions)

assert len(expected) == 43
assert expected["ID"].is_unique
assert expected["Benchmark URI"].is_unique

# 2. Đọc câu hỏi đã sử dụng khi sinh SQL/SPARQL
generated = pd.read_csv(
    run_files[0],
    keep_default_na=False
)

generated["ID"] = generated["ID"].astype(int)
generated["Method"] = generated["Method"].str.strip().str.upper()

assert len(generated) == 86

required = {"ID", "Method", "Inquiry URI", "Question"}
assert required.issubset(generated.columns)

# 3. Đối chiếu theo ID
comparison = generated[
    ["ID", "Method", "Inquiry URI", "Question"]
].merge(
    expected,
    on="ID",
    how="inner",
    validate="many_to_one"
)

assert len(comparison) == 86

def clean(text):
    return " ".join(str(text).split()).casefold()

comparison["URI Match"] = (
    comparison["Inquiry URI"].astype(str).str.strip()
    == comparison["Benchmark URI"].astype(str).str.strip()
)

comparison["Question Match"] = (
    comparison["Question"].map(clean)
    == comparison["Benchmark Question"].map(clean)
)

# 4. Hiển thị sai lệch nếu có
mismatches = comparison[
    ~comparison["URI Match"] | ~comparison["Question Match"]
]

print("=== BENCHMARK ALIGNMENT ===")
print("Total:", len(comparison))
print("URI matched:", int(comparison["URI Match"].sum()))
print("Question matched:", int(comparison["Question Match"].sum()))

if not mismatches.empty:
    display(
        mismatches[
            ["ID", "Method", "Inquiry URI",
             "Benchmark URI", "Question", "Benchmark Question"]
        ]
    )

assert mismatches.empty, (
    "ID/URI/Question không khớp benchmark gốc"
)

# 5. Kiểm tra kết quả AOEA và Quadrant
oea = pd.read_csv(EVAL_DIR / "oea_per_question.csv")
aoea = pd.read_csv(EVAL_DIR / "aoea_summary.csv")
by_quadrant = pd.read_csv(EVAL_DIR / "oea_by_quadrant.csv")

assert len(oea) == 86
assert len(by_quadrant) == 86
assert by_quadrant["Quadrant"].nunique() == 4

for method in ["SQL", "SPARQL"]:
    overall = oea.loc[
        oea["Method"] == method, "OEA"
    ].mean()

    saved = aoea.loc[
        aoea["Method"] == method, "AOEA"
    ].iloc[0]

    quadrant_overall = by_quadrant.loc[
        by_quadrant["Method"] == method, "OEA"
    ].mean()

    assert abs(overall - saved) < 1e-10
    assert abs(overall - quadrant_overall) < 1e-10

    print(f"{method} AOEA: {overall * 100:.2f}%")

print("\nBenchmark alignment: OK")
print("AOEA consistency: OK")
print("Evaluation validation: OK")

=== BENCHMARK ALIGNMENT ===
Total: 86
URI matched: 86
Question matched: 86
SQL AOEA: 45.72%
SPARQL AOEA: 65.53%

Benchmark alignment: OK
AOEA consistency: OK
Evaluation validation: OK
